# NBA Box-Score Scraper — Kaggle (free, no spend)

Pulls every 2025-26 NBA game's `BoxScoreTraditionalV2` via `nba_api` (free public NBA stats API).

Each game gets per-player MIN/PTS/REB/AST + DNP list with reason ('DND - Injury', 'DNP - Coach's Decision', etc).
**Leakage-safe**: only data from THAT game's actual box score, no future trades or current injuries.

**Setup before running:**
1. Settings → Internet ON
2. Add-ons → Secrets → Add: `HF_TOKEN` = `<your LBJLincoln26 owner token from local .env.local HF_TOKEN_NBA>`
3. Run all

**Time:** ~12-14 min for 1257 games (rate-limited at 0.6s/call). CPU-only is fine.

In [ ]:
!pip install --quiet nba_api huggingface_hub
import os
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF_TOKEN set, len=', len(HF_TOKEN))

In [ ]:
# Pull games-2025-26.json from the NBA TF Space
from huggingface_hub import hf_hub_download
import json
GAMES_PATH = hf_hub_download(
    repo_id='LBJLincoln26/nba-llm-trading-floor',
    filename='data/games-2025-26.json',
    repo_type='space', token=HF_TOKEN,
)
doc = json.load(open(GAMES_PATH))
games = doc.get('games', doc) if isinstance(doc, dict) else doc
print(f'loaded {len(games)} games')

In [ ]:
# Scrape every game's BoxScoreTraditionalV2
import time
from nba_api.stats.endpoints import boxscoretraditionalv2

def _row_to_dict(row):
    m_raw = row.get('MIN')
    if m_raw is None or (isinstance(m_raw, float) and m_raw != m_raw):
        min_dec = 0.0
    elif isinstance(m_raw, str) and ':' in m_raw:
        try:
            mm, ss = m_raw.split(':')
            min_dec = round(int(mm) + int(ss)/60.0, 1)
        except: min_dec = 0.0
    else:
        try: min_dec = float(m_raw or 0)
        except: min_dec = 0.0
    return {
        'name': row.get('PLAYER_NAME', '?'),
        'min': min_dec,
        'pts': int(row.get('PTS') or 0),
        'reb': int(row.get('REB') or 0),
        'ast': int(row.get('AST') or 0),
        'comment': (row.get('COMMENT') or '')[:60],
    }

out = {}
failures = []
for i, g in enumerate(games):
    gid = g.get('game_id', '')
    if not gid: continue
    date = (g.get('game_date') or '')[:10]
    h_obj = g.get('home', {})
    a_obj = g.get('away', {})
    home = (h_obj.get('team_abbr') if isinstance(h_obj, dict) else '') or ''
    away = (a_obj.get('team_abbr') if isinstance(a_obj, dict) else '') or ''
    if not (home and away):
        m = (g.get('matchup') or '').replace(' ', '')
        if '@' in m: away, home = m.split('@', 1)
    if not (home and away):
        failures.append(f'{gid}: no abbrs'); continue
    try:
        box = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=gid).get_data_frames()
        pdf = box[0]
        home_rows = pdf[pdf['TEAM_ABBREVIATION'] == home]
        away_rows = pdf[pdf['TEAM_ABBREVIATION'] == away]
        hp = [_row_to_dict(r) for _, r in home_rows.iterrows()]
        ap = [_row_to_dict(r) for _, r in away_rows.iterrows()]
        out[gid] = {
            'date': date, 'home': home, 'away': away,
            'active_home': [p for p in hp if p['min'] > 0],
            'active_away': [p for p in ap if p['min'] > 0],
            'dnp_home':    [p for p in hp if p['min'] == 0],
            'dnp_away':    [p for p in ap if p['min'] == 0],
        }
    except Exception as e:
        failures.append(f'{gid}: {str(e)[:80]}')
        print(f'  [{i+1}/{len(games)}] {gid} FAIL: {e}')
        continue
    if (i+1) % 50 == 0:
        print(f'  [{i+1}/{len(games)}] scraped {len(out)}')
    time.sleep(0.6)

print(f'\n=== scraped {len(out)} games, {len(failures)} failures ===')

In [ ]:
# Save + push to HF dataset + NBA TF Space
import json, os
OUT = '/kaggle/working/box-scores-2025-26.json'
with open(OUT, 'w') as f: json.dump(out, f)
print(f'wrote {OUT}, {os.path.getsize(OUT)/1024/1024:.1f} MB')

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

# Archive dataset
api.create_repo('LBJLincoln26/nba-box-scores', repo_type='dataset', private=False, exist_ok=True)
api.upload_file(
    path_or_fileobj=OUT,
    path_in_repo='box-scores-2025-26.json',
    repo_id='LBJLincoln26/nba-box-scores', repo_type='dataset',
    commit_message=f'[kaggle-box-scrape] {len(out)} games via nba_api',
)

# NBA TF Space — so app.py reads it locally
api.upload_file(
    path_or_fileobj=OUT,
    path_in_repo='data/box-scores-2025-26.json',
    repo_id='LBJLincoln26/nba-llm-trading-floor', repo_type='space',
    commit_message=f'[kaggle-box-scrape] {len(out)} leakage-safe per-game lineups',
)
print('pushed to LBJLincoln26/nba-box-scores + LBJLincoln26/nba-llm-trading-floor')

## Done.

Verify:
- https://huggingface.co/datasets/LBJLincoln26/nba-box-scores
- https://huggingface.co/spaces/LBJLincoln26/nba-llm-trading-floor/tree/main/data (look for box-scores-2025-26.json)

Then restart NBA TF Space — agents will read the new file and see real per-game lineups + DNPs.